# Laboratorio 1 — Análisis del dataset semanal de Corporación Favorita

## Diplomado en Data Engineer

En este laboratorio se trabajará con el dataset semanal agregado de Corporación Favorita.  
Cada registro representa la actividad de un producto específico, en una tienda específica, durante una semana determinada.

La unidad de análisis es:

**Año  × Semana  × Tienda × Producto**

El propósito del laboratorio es comprender la estructura del dataset, evaluar su calidad, analizar las ventas y estudiar el posible aporte de las variables asociadas al precio del petróleo.



## Objetivos de aprendizaje

Al finalizar este laboratorio, el estudiante será capaz de:

- Cargar eficientemente un archivo Parquet.
- Identificar la unidad de análisis de un dataset empresarial.
- Evaluar dimensiones, tipos de datos, memoria, valores nulos y duplicados.
- Analizar la distribución de las ventas semanales.
- Interpretar variables temporales y variables exógenas.
- Evaluar preliminarmente la relación entre ventas y precio del petróleo.
- Construir variables temporales básicas.
- Validar que no se altere la granularidad del dataset.
- Guardar una versión limpia y documentada en formato Parquet.

## 1. Configuración del entorno

Primero se importan las bibliotecas necesarias.

Se utilizarán principalmente:

- `pandas` para manipulación de datos.
- `numpy` para operaciones numéricas.
- `plotly` para construir gráficos interactivos.
- `pyarrow` para trabajar eficientemente con archivos Parquet.

In [1]:
# Instalación de dependencias necesarias en Google Colab
!pip install -q pyarrow plotly statsmodels

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 2. Carga del archivo

En Google Colab existen distintas alternativas para cargar el archivo:

1. Subirlo directamente desde el computador.
2. Leerlo desde Google Drive.
3. Leerlo desde una ruta previamente conocida.


## 3. Lectura eficiente del archivo Parquet

El formato Parquet permite almacenar grandes volúmenes de datos con compresión y preservando los tipos de datos.

A diferencia de un archivo CSV, Parquet normalmente:

- ocupa menos espacio;
- se carga más rápido;
- conserva tipos numéricos y temporales;
- permite seleccionar columnas específicas al momento de lectura.

En esta primera lectura se cargarán todas las columnas para realizar una inspección general.

In [ ]:
ruta_archivo = Path("/content/favorita_weekly_oil.parquet")

df = pd.read_parquet(ruta_archivo, engine="pyarrow")

print("Archivo leído correctamente.")
print("Dimensiones :", df.shape)

## 4. Inspección inicial

Antes de modificar los datos se debe observar su estructura general.

Se revisarán:

- primeras filas;
- últimas filas;
- muestra aleatoria;
- nombres de columnas;
- tipos de datos;
- cantidad de registros y variables.

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(5)

In [ ]:
print("Cantidad de filas      :", df.shape[0])
print("Cantidad de columnas   :", df.shape[1])
print("\nColumnas disponibles :")
for columna in df.columns:
    print("   ->", columna)

In [ ]:
df.info(memory_usage="deep")

## 5. Verificación de las columnas esperadas

El dataset semanal debería contener las siguientes variables:

- `fecha_semana`
- `anio`
- `semana`
- `store_nbr`
- `item_nbr`
- `unit_sales`
- `dias_promocion`
- `dias_con_venta`
- `oil_promedio_semana`
- `oil_min_semana`
- `oil_max_semana`

Esta validación permite detectar errores de nombre, columnas omitidas o cambios en la estructura del archivo.

In [ ]:
columnas_esperadas = [
    "fecha_semana",
    "anio",
    "semana",
    "store_nbr",
    "item_nbr",
    "unit_sales",
    "dias_promocion",
    "dias_con_venta",
    "oil_promedio_semana",
    "oil_min_semana",
    "oil_max_semana"
]

columnas_faltantes   = sorted(set(columnas_esperadas) - set(df.columns))
columnas_adicionales = sorted(set(df.columns) - set(columnas_esperadas))

print("Columnas faltantes   : ", columnas_faltantes)
print("Columnas adicionales : ", columnas_adicionales)

if not columnas_faltantes:
    print("\nLa estructura contiene todas las columnas esperadas.")

## 6. Diccionario inicial de variables

Antes del análisis, conviene clasificar las variables según su función analítica.

Los identificadores numéricos, como tienda y producto, no deben interpretarse automáticamente como variables continuas.  
Aunque estén almacenados como números, su función principal es identificar entidades.

In [ ]:
diccionario = pd.DataFrame({
    "variable": [
        "fecha_semana", "anio", "semana", "store_nbr", "item_nbr",
        "unit_sales", "dias_promocion", "dias_con_venta",
        "oil_promedio_semana", "oil_min_semana", "oil_max_semana"
    ],
    "descripcion": [
        "Fecha de referencia de la semana",
        "Año según calendario ISO",
        "Número de semana ISO",
        "Identificador de tienda",
        "Identificador de producto",
        "Ventas unitarias semanales",
        "Cantidad de días con promoción",
        "Cantidad de días con ventas",
        "Precio promedio semanal del petróleo",
        "Precio mínimo semanal del petróleo",
        "Precio máximo semanal del petróleo"
    ],
    "nivel_medicion": [
        "Temporal", "Ordinal temporal", "Ordinal temporal",
        "Identificador", "Identificador", "Numérica",
        "Numérica discreta", "Numérica discreta",
        "Numérica continua", "Numérica continua", "Numérica continua"
    ],
    "rol_preliminar_ml": [
        "Predictora temporal", "Predictora temporal", "Predictora temporal",
        "Identificador/categórica", "Identificador/categórica",
        "Posible objetivo", "Predictora",
        "Predictora con riesgo de disponibilidad",
        "Predictora exógena", "Predictora exógena", "Predictora exógena"
    ]
})

display(diccionario)

## 7. Revisión y conversión de tipos de datos

Los tipos de datos influyen directamente en:

- consumo de memoria;
- velocidad de procesamiento;
- compatibilidad con algoritmos;
- interpretación de las variables.

Se recomienda:

- convertir `fecha_semana` a fecha;
- usar enteros pequeños para año, semana y conteos;
- mantener ventas y petróleo como variables numéricas;
- tratar tienda y producto como identificadores, aunque se almacenen mediante enteros.

In [ ]:
df["fecha_semana"] = pd.to_datetime(df["fecha_semana"], errors="coerce")

conversiones_enteros = {
    "anio": "Int16",
    "semana": "Int8",
    "store_nbr": "Int16",
    "item_nbr": "Int32",
    "dias_promocion": "Int8",
    "dias_con_venta": "Int8"
}

for columna, tipo in conversiones_enteros.items():
    if columna in df.columns:
        df[columna] = pd.to_numeric(df[columna], errors="coerce").astype(tipo)

columnas_float = [
    "unit_sales",
    "oil_promedio_semana",
    "oil_min_semana",
    "oil_max_semana"
]

for columna in columnas_float:
    if columna in df.columns:
        df[columna] = pd.to_numeric(df[columna], errors="coerce").astype("float32")

df.info(memory_usage="deep")

### Construcción de la semana correlativa

La variable `semana_correlativa` asigna un número secuencial a cada fecha semanal del dataset. A diferencia de `semana`, que vuelve a comenzar en 1 cuando cambia el año, esta variable aumenta continuamente. Se utilizará en los gráficos interactivos para identificar con claridad cada observación temporal.

In [ ]:
fechas_ordenadas = np.sort(df["fecha_semana"].dropna().unique())

mapa_semana_correlativa = {
    fecha: numero
    for numero, fecha in enumerate(fechas_ordenadas, start=1)
}

df["semana_correlativa"] = (
    df["fecha_semana"]
    .map(mapa_semana_correlativa)
    .astype("Int16")
)

display(
    df[["fecha_semana", "anio", "semana", "semana_correlativa"]]
    .drop_duplicates()
    .sort_values("fecha_semana")
    .head(15)
)

## 8. Medición del consumo de memoria

En datasets con millones de registros, el consumo de memoria es un criterio relevante.

La optimización de tipos debe realizarse sin perder información ni precisión necesaria para el análisis.

In [ ]:
memoria_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memoria utilizada por el DataFrame: {memoria_mb:,.2f} MB")

memoria_columnas = (
    df.memory_usage(deep=True)
      .sort_values(ascending=False)
      .div(1024**2)
      .rename("memoria_mb")
      .reset_index()
      .rename(columns={"index": "variable"})
)

memoria_columnas

## 9. Análisis de valores nulos

Los valores nulos deben evaluarse antes del modelamiento.

No todos los nulos tienen el mismo significado:

- Un nulo en ventas puede indicar un error de construcción.
- Un nulo en petróleo puede originarse en días o semanas sin registro.
- Un nulo en fecha o identificadores puede impedir mantener la unidad de análisis.

Se calculará la cantidad y el porcentaje de nulos por variable.

In [ ]:
reporte_nulos = pd.DataFrame({
    "cantidad_nulos"  : df.isna().sum(),
    "porcentaje_nulos": df.isna().mean() * 100
}).sort_values("porcentaje_nulos", ascending=False)

reporte_nulos

## 10. Visualización de valores nulos

El gráfico permite reconocer rápidamente cuáles variables concentran problemas de completitud.

In [ ]:
nulos_grafico = (
    reporte_nulos[reporte_nulos["cantidad_nulos"] > 0]
    .reset_index()
    .rename(columns={"index": "variable"})
)

if len(nulos_grafico) > 0:
    fig = px.bar(
        nulos_grafico,
        x="variable",
        y="porcentaje_nulos",
        title="Valores nulos por variable",
        labels={
            "variable": "Variable",
            "porcentaje_nulos": "Porcentaje de valores nulos"
        },
        hover_data=["cantidad_nulos"]
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print("No se detectaron valores nulos.")

## 11. Tratamiento preliminar de valores nulos

En este laboratorio no se deben imputar automáticamente todos los nulos.

Se aplicarán criterios conservadores:

- Filas sin fecha, tienda, producto o ventas se separarán para revisión.
- Los nulos del petróleo se conservarán inicialmente.
- La imputación del petróleo deberá realizarse utilizando únicamente información temporal válida.

En problemas predictivos, cualquier imputación debe aprenderse con datos de entrenamiento para evitar fuga de información.

In [ ]:
columnas_criticas = ["fecha_semana", "store_nbr", "item_nbr", "unit_sales"]

mascara_criticos      = df[columnas_criticas].isna().any(axis=1)
df_registros_criticos = df.loc[mascara_criticos].copy()

print("Registros con nulos en columnas críticas:", len(df_registros_criticos))
display(df_registros_criticos.head())

In [ ]:
# Eliminación conservadora de registros que no permiten identificar
# la unidad de análisis o la variable de ventas.
df = df.dropna(subset=columnas_criticas).copy()

print("Dimensiones después de eliminar registros críticos:", df.shape)

## 12. Validación de la unidad de análisis

La unidad de análisis es:

**Año × Semana × Tienda × Producto**

No debería existir más de un registro para una misma combinación.

La existencia de duplicados podría indicar:

- un problema de agregación;
- una unión incorrecta;
- una repetición de registros;
- inconsistencia entre la fecha y las variables ISO.

In [ ]:
llave_unidad_analisis = ["anio", "semana", "store_nbr", "item_nbr"]

duplicados = df.duplicated(subset=llave_unidad_analisis, keep=False)
cantidad_duplicados = duplicados.sum()

print("Filas involucradas en duplicados:", cantidad_duplicados)
print("Porcentaje:", cantidad_duplicados / len(df) * 100)

display(
    df.loc[duplicados]
      .sort_values(llave_unidad_analisis)
      .head(20)
)

## 13. Consistencia entre fecha, año ISO y semana ISO

La fecha puede utilizarse para recalcular el año y la semana ISO.

Esta comparación permite detectar inconsistencias entre:

- `fecha_semana`;
- `anio`;
- `semana`.

In [ ]:
calendario_iso = df["fecha_semana"].dt.isocalendar()

df["anio_iso_calculado"]   = calendario_iso["year"].astype("Int16")
df["semana_iso_calculada"] = calendario_iso["week"].astype("Int8")

inconsistencia_anio   = df["anio"] != df["anio_iso_calculado"]
inconsistencia_semana = df["semana"] != df["semana_iso_calculada"]

print("Inconsistencias de año ISO    :", inconsistencia_anio.sum())
print("Inconsistencias de semana ISO :", inconsistencia_semana.sum())

display(
    df.loc[inconsistencia_anio | inconsistencia_semana,
           ["fecha_semana", "anio", "semana",
            "anio_iso_calculado", "semana_iso_calculada"]]
      .head(20)
)

## 14. Validaciones de rango

Algunas variables tienen rangos naturales:

- Semana ISO: normalmente entre 1 y 53.
- Días de promoción: entre 0 y 7.
- Días con venta: entre 0 y 7.
- Precio mínimo del petróleo no debería superar al promedio.
- Precio promedio no debería superar al máximo.

Estas reglas ayudan a identificar valores imposibles o inconsistentes.

In [ ]:
validaciones_rango = {
    "semana_fuera_rango"        : ~df["semana"].between(1, 53),
    "dias_promocion_fuera_rango": ~df["dias_promocion"].between(0, 7),
    "dias_con_venta_fuera_rango": ~df["dias_con_venta"].between(0, 7),
    "oil_min_mayor_promedio"    : df["oil_min_semana"] > df["oil_promedio_semana"],
    "oil_promedio_mayor_max"    : df["oil_promedio_semana"] > df["oil_max_semana"],
    "oil_min_mayor_max"         : df["oil_min_semana"] > df["oil_max_semana"]
}

reporte_validaciones = pd.DataFrame({
    nombre: [mascara.sum(), mascara.mean() * 100]
    for nombre, mascara in validaciones_rango.items()
}, index=["cantidad", "porcentaje"]).T

reporte_validaciones

## 15. Cardinalidad de las variables

La cardinalidad corresponde a la cantidad de valores distintos de una variable.

Es especialmente relevante para:

- tiendas;
- productos;
- fechas;
- variables categóricas;
- decisiones de codificación.

Un identificador de producto puede tener una cardinalidad muy alta, por lo que aplicar One-Hot Encoding directamente podría ser poco eficiente.

In [ ]:
cardinalidad = pd.DataFrame({
    "variable": df.columns,
    "valores_unicos": [df[col].nunique(dropna=True) for col in df.columns],
    "porcentaje_unicidad": [
        df[col].nunique(dropna=True) / len(df) * 100 for col in df.columns
    ]
}).sort_values("valores_unicos", ascending=False)

cardinalidad




In [ ]:
# Una vez analizada la información, se libera memoria
del cardinalidad

import gc
gc.collect()

## 16. Estadísticas descriptivas

Las estadísticas descriptivas permiten reconocer:

- escala de las variables;
- valores mínimos y máximos;
- dispersión;
- posibles valores extremos;
- diferencias entre variables de conteo y continuas.

La interpretación debe considerar que tienda y producto son identificadores, aunque aparezcan en el resumen numérico.

In [ ]:
columnas_analiticas = [
    "unit_sales",
    "dias_promocion",
    "dias_con_venta",
    "oil_promedio_semana",
    "oil_min_semana",
    "oil_max_semana"
]

df[columnas_analiticas].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

## 17. Análisis de la variable objetivo: ventas semanales

`unit_sales` puede utilizarse como variable objetivo en problemas de regresión.

Antes del modelamiento se debe estudiar:

- presencia de valores negativos;
- concentración de ceros;
- asimetría;
- valores extremos.

En el dataset original de Corporación Favorita, valores negativos pueden representar devoluciones.  
Por lo tanto, no deben eliminarse sin analizar su significado empresarial.

In [ ]:
ventas = df["unit_sales"]

resumen_ventas = pd.Series({
    "cantidad_registros": len(ventas),
    "ventas_negativas": (ventas < 0).sum(),
    "ventas_iguales_cero": (ventas == 0).sum(),
    "ventas_positivas": (ventas > 0).sum(),
    "porcentaje_negativas": (ventas < 0).mean() * 100,
    "porcentaje_ceros": (ventas == 0).mean() * 100,
    "media": ventas.mean(),
    "mediana": ventas.median(),
    "desviacion_estandar": ventas.std(),
    "minimo": ventas.min(),
    "maximo": ventas.max()
})

resumen_ventas.to_frame("valor")

In [ ]:
del resumen_ventas

import gc
gc.collect()

## 18. Evolución temporal de las ventas

Para obtener una visión general, se agregarán las ventas por semana.

Esta agregación es solo para análisis descriptivo.  
No reemplaza la unidad de análisis original del dataset.

In [ ]:
ventas_semanales = (
    df.groupby("fecha_semana", as_index=False)
      .agg(
          semana_correlativa=("semana_correlativa", "first"),
          ventas_totales=("unit_sales", "sum"),
          ventas_promedio=("unit_sales", "mean"),
          productos_tienda=("unit_sales", "size")
      )
      .sort_values("fecha_semana")
)

display(ventas_semanales.head())

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ventas_semanales["fecha_semana"],
    y=ventas_semanales["ventas_totales"],
    mode="lines+markers",
    name="Ventas totales",
    customdata=ventas_semanales[["semana_correlativa"]],
    hovertemplate=(
        "<b>Fecha:</b> %{x|%d-%m-%Y}<br>"
        "<b>Semana correlativa:</b> %{customdata[0]}<br>"
        "<b>Ventas totales:</b> %{y:,.2f}"
        "<extra></extra>"
    )
))
fig.update_layout(
    title="Evolución semanal de las ventas totales",
    xaxis_title="Fecha",
    yaxis_title="Ventas totales",
    hovermode="x unified"
)
fig.show()

## 19. Comprensión de las variables del petróleo

Las variables del petróleo son variables exógenas, es decir, se originan fuera de la operación directa de la tienda.

En el contexto ecuatoriano, el precio del petróleo puede relacionarse indirectamente con:

- actividad económica;
- empleo;
- ingreso disponible;
- inflación;
- transporte y logística;
- consumo de los hogares.

Una correlación no implica causalidad.  
El objetivo inicial es determinar si estas variables aportan información temporal útil.

In [ ]:
columnas_petroleo = [
    "oil_promedio_semana",
    "oil_min_semana",
    "oil_max_semana"
]

display(df[columnas_petroleo].describe().T)

## 20. Coherencia interna del petróleo

Se verificará que se cumpla:

- mínimo ≤ promedio;
- promedio ≤ máximo.

Además, se construirá el rango semanal del petróleo como diferencia entre máximo y mínimo.

In [ ]:
df["oil_rango_semana"] = df["oil_max_semana"] - df["oil_min_semana"]

print("Registros con rango negativo:", (df["oil_rango_semana"] < 0).sum())
display(df[["oil_min_semana", "oil_promedio_semana", "oil_max_semana", "oil_rango_semana"]].head())

## 21. Serie temporal del precio del petróleo

Como el valor del petróleo se repite para todas las combinaciones tienda-producto de una misma semana, se debe obtener una sola observación por fecha antes de graficar.

Utilizar todas las filas produciría una visualización redundante y podría sesgar algunos cálculos.

In [ ]:
petroleo_semanal = (
    df.groupby("fecha_semana", as_index=False)
      .agg(
          semana_correlativa=("semana_correlativa", "first"),
          oil_promedio_semana=("oil_promedio_semana", "first"),
          oil_min_semana=("oil_min_semana", "first"),
          oil_max_semana=("oil_max_semana", "first"),
          oil_rango_semana=("oil_rango_semana", "first")
      )
      .sort_values("fecha_semana")
)

display(petroleo_semanal.head())

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=petroleo_semanal["fecha_semana"],
        y=petroleo_semanal["oil_promedio_semana"],
        mode="lines+markers",
        name="Precio promedio"
    )
)

fig.add_trace(
    go.Scatter(
        x=petroleo_semanal["fecha_semana"],
        y=petroleo_semanal["oil_max_semana"],
        mode="lines",
        name="Precio máximo",
        line=dict(dash="dot")
    )
)

fig.add_trace(
    go.Scatter(
        x=petroleo_semanal["fecha_semana"],
        y=petroleo_semanal["oil_min_semana"],
        mode="lines",
        name="Precio mínimo",
        line=dict(dash="dot")
    )
)

fig.update_layout(
    title="Evolución semanal del precio del petróleo",
    xaxis_title="Fecha",
    yaxis_title="Precio semanal",
    hovermode="x unified"
)

fig.show()

## 22. Comparación temporal entre ventas y petróleo

Para comparar ambas series, se unirán las ventas agregadas por semana con el precio promedio del petróleo.

Es importante usar una única observación semanal del petróleo para evitar ponderarlo artificialmente por la cantidad de productos o tiendas.

In [ ]:
serie_ventas_petroleo = ventas_semanales.merge(
    petroleo_semanal,
    on="fecha_semana",
    how="left",
    validate="one_to_one"
)

display(serie_ventas_petroleo.head())

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=serie_ventas_petroleo["fecha_semana"],
    y=serie_ventas_petroleo["ventas_totales"],
    name="Ventas totales",
    mode="lines+markers",
    yaxis="y",
    customdata=serie_ventas_petroleo[["semana_correlativa_x", "oil_promedio_semana"]],
    hovertemplate=(
        "<b>Fecha:</b> %{x|%d-%m-%Y}<br>"
        "<b>Semana correlativa:</b> %{customdata[0]}<br>"
        "<b>Ventas totales:</b> %{y:,.2f}<br>"
        "<b>Precio petróleo:</b> %{customdata[1]:,.2f}"
        "<extra></extra>"
    )
))

fig.add_trace(go.Scatter(
    x=serie_ventas_petroleo["fecha_semana"],
    y=serie_ventas_petroleo["oil_promedio_semana"],
    name="Precio promedio del petróleo",
    mode="lines+markers",
    yaxis="y2",
    customdata=serie_ventas_petroleo[["semana_correlativa_y", "ventas_totales"]],
    hovertemplate=(
        "<b>Fecha:</b> %{x|%d-%m-%Y}<br>"
        "<b>Semana correlativa:</b> %{customdata[0]}<br>"
        "<b>Precio petróleo:</b> %{y:,.2f}<br>"
        "<b>Ventas totales:</b> %{customdata[1]:,.2f}"
        "<extra></extra>"
    )
))

fig.update_layout(
    title="Ventas totales y precio promedio del petróleo",
    xaxis=dict(title="Fecha"),
    yaxis=dict(title="Ventas totales"),
    yaxis2=dict(title="Precio promedio del petróleo", overlaying="y", side="right"),
    hovermode="x unified",
    legend=dict(orientation="h", y=1.12)
)
fig.show()

## 23. Correlación entre ventas y petróleo

La correlación mide asociación lineal, pero no permite concluir que una variable cause la otra.

Además, las series temporales pueden compartir tendencias y estacionalidad, generando correlaciones engañosas.

Por eso este resultado debe interpretarse como exploratorio.

In [ ]:
correlaciones = serie_ventas_petroleo[
    [
        "ventas_totales",
        "ventas_promedio",
        "oil_promedio_semana",
        "oil_min_semana",
        "oil_max_semana",
        "oil_rango_semana"
    ]
].corr()

display(correlaciones)

## 24. Relación gráfica entre petróleo y ventas

El diagrama de dispersión permite observar si existe una relación aproximadamente lineal, grupos, cambios de régimen o valores extremos.

In [ ]:
fig = px.scatter(
    serie_ventas_petroleo,
    x="oil_promedio_semana",
    y="ventas_totales",
    custom_data=["fecha_semana", "semana_correlativa_x"],
    title="Relación entre petróleo y ventas totales",
    labels={
        "oil_promedio_semana": "Precio promedio semanal del petróleo",
        "ventas_totales": "Ventas totales semanales"
    },
    trendline="ols"
)

fig.update_traces(
    hovertemplate=(
        "<b>Fecha:</b> %{customdata[0]|%d-%m-%Y}<br>"
        "<b>Semana correlativa:</b> %{customdata[1]}<br>"
        "<b>Ventas totales:</b> %{y:,.2f}<br>"
        "<b>Precio petróleo:</b> %{x:,.2f}"
        "<extra></extra>"
    ),
    selector=dict(mode="markers")
)
fig.show()

## 25. Ingeniería de variables temporales

A partir de `fecha_semana` se pueden construir variables útiles para representar estacionalidad.

Se crearán:

- mes;
- trimestre;
- año calendario;
- semana ISO;
- inicio y fin de año;
- componentes seno y coseno de la semana.

Las representaciones seno y coseno permiten expresar que la semana 52 y la semana 1 son temporalmente cercanas.

In [ ]:
df["mes"] = df["fecha_semana"].dt.month.astype("Int8")
df["trimestre"] = df["fecha_semana"].dt.quarter.astype("Int8")
df["anio_calendario"] = df["fecha_semana"].dt.year.astype("Int16")

df["es_inicio_anio"] = df["semana"].isin([1, 2, 3]).astype("Int8")
df["es_fin_anio"] = df["semana"].isin([50, 51, 52, 53]).astype("Int8")

df["semana_seno"] = np.sin(2 * np.pi * df["semana"].astype(float) / 52).astype("float32")
df["semana_coseno"] = np.cos(2 * np.pi * df["semana"].astype(float) / 52).astype("float32")

display(
    df[
        [
            "fecha_semana", "anio", "semana", "mes", "trimestre",
            "es_inicio_anio", "es_fin_anio",
            "semana_seno", "semana_coseno"
        ]
    ].head()
)

## 26. Variables históricas y riesgo de fuga de información

Para predecir ventas futuras suele ser útil crear rezagos y promedios móviles, por ejemplo:

- ventas de la semana anterior;
- promedio de las últimas cuatro semanas;
- máximo de las últimas ocho semanas.

Estas variables deben calcularse respetando el orden temporal de cada combinación tienda-producto.

El uso de `shift(1)` garantiza que la venta de la semana actual no se utilice para predecirse a sí misma.

In [ ]:
df = df.sort_values(
    ["store_nbr", "item_nbr", "fecha_semana"]
).copy()

grupo_producto_tienda = df.groupby(
    ["store_nbr", "item_nbr"],
    observed=True
)["unit_sales"]

df["ventas_lag_1"] = grupo_producto_tienda.shift(1).astype("float32")
df["ventas_lag_2"] = grupo_producto_tienda.shift(2).astype("float32")
df["ventas_lag_4"] = grupo_producto_tienda.shift(4).astype("float32")

df["ventas_media_4_sem"] = (
    grupo_producto_tienda
    .shift(1)
    .rolling(window=4, min_periods=1)
    .mean()
    .astype("float32")
)

display(
    df[
        [
            "store_nbr", "item_nbr", "fecha_semana", "unit_sales",
            "ventas_lag_1", "ventas_lag_2", "ventas_lag_4",
            "ventas_media_4_sem"
        ]
    ].head(15)
)

## 27. Advertencia sobre las variables históricas

Los rezagos anteriores son válidos solo si el dataset está correctamente ordenado y no existen semanas faltantes inesperadas.

Además:

- deben construirse después de definir el horizonte de predicción;
- no deben incorporar semanas futuras;
- en validación cruzada temporal deben recalcularse respetando cada corte;
- no deben utilizar información del conjunto de prueba para transformar entrenamiento.

En un laboratorio posterior se implementará este procedimiento dentro de un flujo de modelamiento temporal.

## 28. Análisis por tienda

Se calcularán ventas agregadas por tienda.

Este análisis permite identificar:

- tiendas de mayor volumen;
- heterogeneidad operacional;
- posibles tiendas atípicas;
- diferencias que luego podrán explicarse con `stores.csv`.

In [ ]:
ventas_por_tienda = (
    df.groupby("store_nbr", as_index=False)
      .agg(
          ventas_totales=("unit_sales", "sum"),
          ventas_promedio=("unit_sales", "mean"),
          productos_distintos=("item_nbr", "nunique"),
          semanas=("fecha_semana", "nunique")
      )
      .sort_values("ventas_totales", ascending=False)
)

display(ventas_por_tienda.head(20))

In [ ]:
ventas_por_tienda["store_nbr_texto"] = ventas_por_tienda["store_nbr"].astype(str)

fig = px.bar(
    ventas_por_tienda,
    x="store_nbr_texto",
    y="ventas_totales",
    title="Ventas totales por tienda",
    labels={
        "store_nbr_texto": "Tienda",
        "ventas_totales": "Ventas totales"
    },
    hover_data=["ventas_promedio", "productos_distintos", "semanas"]
)

fig.update_layout(xaxis_tickangle=-90)
fig.show()

## 29. Análisis por producto

Se identificarán los productos con mayores ventas acumuladas.

El identificador `item_nbr` permite reconocer el producto, pero aún no entrega información sobre familia, clase o condición perecible.  
Esa información se incorporará posteriormente desde `items.csv`.

In [ ]:
ventas_por_producto = (
    df.groupby("item_nbr", as_index=False)
      .agg(
          ventas_totales=("unit_sales", "sum"),
          ventas_promedio=("unit_sales", "mean"),
          tiendas=("store_nbr", "nunique"),
          semanas=("fecha_semana", "nunique")
      )
      .sort_values("ventas_totales", ascending=False)
)

display(ventas_por_producto.head(20))

## 30. Reporte automatizado de calidad

Se construirá una función que reúna los principales indicadores de calidad del dataset.

Este reporte puede reutilizarse después de integrar nuevos archivos.

In [ ]:
def generar_reporte_calidad(
    datos: pd.DataFrame,
    llave: list[str]
) -> pd.DataFrame:
    memoria_mb = datos.memory_usage(deep=True).sum() / 1024**2

    indicadores = {
        "filas": len(datos),
        "columnas": datos.shape[1],
        "duplicados_completos": datos.duplicated().sum(),
        "duplicados_unidad_analisis": datos.duplicated(subset=llave).sum(),
        "celdas_nulas": int(datos.isna().sum().sum()),
        "porcentaje_celdas_nulas": datos.isna().mean().mean() * 100,
        "memoria_mb": memoria_mb,
        "fecha_minima": datos["fecha_semana"].min(),
        "fecha_maxima": datos["fecha_semana"].max(),
        "tiendas_unicas": datos["store_nbr"].nunique(),
        "productos_unicos": datos["item_nbr"].nunique()
    }

    return pd.DataFrame(
        indicadores.items(),
        columns=["indicador", "valor"]
    )

llave_unidad_analisis = ["anio", "semana", "store_nbr", "item_nbr"]
reporte_calidad = generar_reporte_calidad(
    df,
    llave_unidad_analisis
)

display(reporte_calidad)

## 31. Reporte detallado por variable

Este reporte resume:

- tipo de dato;
- cantidad y porcentaje de nulos;
- cardinalidad;
- memoria utilizada.

Será útil para documentar la evolución del dataset.

In [ ]:
def reporte_columnas(datos: pd.DataFrame) -> pd.DataFrame:
    resultado = []

    for columna in datos.columns:
        resultado.append({
            "variable": columna,
            "tipo": str(datos[columna].dtype),
            "nulos": int(datos[columna].isna().sum()),
            "porcentaje_nulos": datos[columna].isna().mean() * 100,
            "valores_unicos": int(datos[columna].nunique(dropna=True)),
            "memoria_mb": datos[columna].memory_usage(deep=True) / 1024**2
        })

    return pd.DataFrame(resultado).sort_values(
        "memoria_mb",
        ascending=False
    )

reporte_variables = reporte_columnas(df)
display(reporte_variables)

## 32. Validación final de granularidad

Antes de guardar el archivo, se debe verificar nuevamente que la unidad de análisis no haya cambiado.

La creación de variables no debería aumentar ni disminuir la cantidad de filas.

In [ ]:
duplicados_finales = df.duplicated(
    subset=llave_unidad_analisis
).sum()

print("Cantidad final de filas:", len(df))
print("Duplicados finales en la unidad de análisis:", duplicados_finales)

if duplicados_finales == 0:
    print("La unidad de análisis se mantiene correctamente.")
else:
    print("Advertencia: existen duplicados que deben investigarse antes de guardar.")

## 33. Selección provisional de columnas

El archivo base conservará:

- identificadores;
- variables temporales originales;
- ventas;
- promociones;
- días con venta;
- variables del petróleo;
- variables temporales derivadas;
- rezagos construidos correctamente.

Las variables `anio_iso_calculado` y `semana_iso_calculada` se utilizaron para validación y pueden eliminarse si las variables originales resultaron consistentes.

In [ ]:
columnas_finales = [
    "fecha_semana",
    "anio",
    "semana",
    "semana_correlativa",
    "store_nbr",
    "item_nbr",
    "unit_sales",
    "dias_promocion",
    "dias_con_venta",
    "oil_promedio_semana",
    "oil_min_semana",
    "oil_max_semana",
    "oil_rango_semana",
    "mes",
    "trimestre",
    "anio_calendario",
    "es_inicio_anio",
    "es_fin_anio",
    "semana_seno",
    "semana_coseno",
    "ventas_lag_1",
    "ventas_lag_2",
    "ventas_lag_4",
    "ventas_media_4_sem"
]

columnas_finales = [
    columna for columna in columnas_finales
    if columna in df.columns
]

df_final = df[columnas_finales].copy()

print("Dimensiones del dataset final:", df_final.shape)
display(df_final.head())

## 34. Guardado del dataset validado

El archivo se guardará en formato Parquet utilizando compresión `snappy`.

Este archivo será la base para los siguientes laboratorios:

`favorita_base.parquet`

In [ ]:
ruta_salida = Path("favorita_base.parquet")

df_final.to_parquet(
    ruta_salida,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print("Archivo guardado:", ruta_salida)
print(f"Tamaño: {ruta_salida.stat().st_size / 1024**2:,.2f} MB")

## 35. Descarga del archivo generado

La siguiente celda permite descargar el archivo desde Google Colab.

In [ ]:
from google.colab import files

files.download("favorita_base.parquet")

# Conclusiones esperadas

El estudiante debería concluir que:

- La preparación de datos no consiste únicamente en ejecutar código.
- La unidad de análisis debe mantenerse durante todo el proceso.
- Los identificadores requieren un tratamiento distinto al de las variables continuas.
- Las variables temporales deben respetar el orden cronológico.
- Las variables históricas pueden generar fuga si se construyen incorrectamente.
- El petróleo es una variable exógena cuya utilidad debe evaluarse empíricamente.
- Una correlación no permite establecer causalidad.
- El dataset maestro puede enriquecerse progresivamente sin perder trazabilidad.